# Inspect chroma-key dataset

Visualize captured frames with the detected yellow-marker corners overlaid in green.

**Usage**: change `RUN_DIR` to point at a `capture_<ts>/` folder, then call `show_grid(start, n=16)` to render `n` frames in a grid starting from frame number `start`. Missing frame numbers are skipped automatically (the function walks forward until it finds `n` valid ones).

Run `inspect_dataset.ipynb` AFTER `extract_quad.py --enrich-json --batch-index ...` has populated `quads_index.json` and per-frame `detected_corners`.

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "chroma_key_dataset_generator" else Path.cwd()
RUN_DIR = REPO_ROOT / "data" / "chroma_key_dataset" / "capture_20260602_211812"

# Load the batch index once (fast) - falls back to per-frame JSON lookup if missing.
INDEX_PATH = RUN_DIR / "quads_index.json"
INDEX = {}
if INDEX_PATH.exists():
    with open(INDEX_PATH) as f:
        INDEX = json.load(f)
    print(f"Loaded {len(INDEX)} corners from quads_index.json")
else:
    print(f"[!] {INDEX_PATH} missing, will read per-frame JSONs instead.")

ALL_PNGS = sorted(RUN_DIR.glob("[0-9][0-9][0-9][0-9][0-9][0-9].png"))
STEMS = [p.stem for p in ALL_PNGS]
print(f"Found {len(STEMS)} png files in {RUN_DIR.name}")

In [ ]:
def get_corners(stem):
    """Return (4, 2) array of corners or None. Tries quads_index first, then per-frame JSON."""
    if stem in INDEX:
        return np.asarray(INDEX[stem]["corners"], dtype=np.float32)
    sidecar = RUN_DIR / f"{stem}.json"
    if not sidecar.exists():
        return None
    with open(sidecar) as f:
        meta = json.load(f)
    if "detected_corners" in meta:
        return np.asarray(meta["detected_corners"]["corners"], dtype=np.float32)
    return None


def load_frame_with_overlay(stem):
    """Load <stem>.png as RGB and draw the detected corners + bbox in green."""
    bgr = cv2.imread(str(RUN_DIR / f"{stem}.png"))
    if bgr is None:
        return None
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    corners = get_corners(stem)
    if corners is None:
        cv2.putText(rgb, "NO QUAD", (10, 40), cv2.FONT_HERSHEY_SIMPLEX,
                    1.0, (255, 80, 80), 2)
        return rgb
    pts = corners.astype(np.int32)
    cv2.polylines(rgb, [pts], isClosed=True, color=(0, 255, 0), thickness=3)
    for x, y in pts:
        cv2.circle(rgb, (int(x), int(y)), 6, (0, 255, 0), -1)
    return rgb


def show_grid(start=1, n=16, cols=4, figsize=(16, 9)):
    """Render `n` frames starting from frame number `start` in a grid.
    Skips missing frame numbers; walks forward until `n` valid frames are found."""
    rows = (n + cols - 1) // cols
    if isinstance(start, str):
        start = int(start)
    # Walk forward from `start` collecting valid PNGs
    chosen = []
    for k in range(start, len(STEMS) * 2 + start):  # generous upper bound
        stem = f"{k:06d}"
        if stem in STEMS:
            chosen.append(stem)
            if len(chosen) == n:
                break
    if not chosen:
        print("No frames found from start.")
        return
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, stem in zip(axes, chosen):
        img = load_frame_with_overlay(stem)
        if img is None:
            ax.text(0.5, 0.5, f"{stem} missing", ha="center")
            continue
        ax.imshow(img)
        ax.set_title(stem, fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect frames 1..16
show_grid(start=1, n=16)

In [ ]:
# Inspect 16 frames starting from #800
show_grid(start=800, n=16)

In [ ]:
# Random batch of 16 from anywhere
import random
random.seed(42)
if STEMS:
    show_grid(start=int(random.choice(STEMS)), n=16)

## Side-by-side triplet comparison

After running `generate_all_datasets.sh` you get three folders sharing the same timestamp:
- `capture_<ts>_marker`
- `capture_<ts>_clean`
- `capture_<ts>_noleader`

Set `RUN_TS` below and use `show_triplets(start, n=4)` to verify the scenes are actually paired (same NPCs, same weather, same camera pose; only the CarlaCola texture changes — or it is absent).

In [ ]:
RUN_TS = "<edit_me>"  # e.g. "20260608_213045"
DATA_ROOT = REPO_ROOT / "data" / "chroma_key_dataset"
MARKER_DIR = DATA_ROOT / f"capture_{RUN_TS}_marker"
CLEAN_DIR = DATA_ROOT / f"capture_{RUN_TS}_clean"
NOLEADER_DIR = DATA_ROOT / f"capture_{RUN_TS}_noleader"

def show_triplets(start=1, n=4, figsize=(18, 9)):
    """Show n rows: marker | clean | noleader, starting from frame `start`."""
    fig, axes = plt.subplots(n, 3, figsize=figsize)
    if n == 1:
        axes = np.array([axes])
    for ax in axes.flatten():
        ax.axis("off")
    marker_stems = sorted(p.stem for p in MARKER_DIR.glob("[0-9]" * 6 + ".png"))
    chosen = []
    for k in range(start, max(start + n * 5, start + 100)):
        stem = f"{k:06d}"
        if stem in marker_stems:
            chosen.append(stem)
            if len(chosen) == n:
                break
    for row, stem in enumerate(chosen):
        for col, d in enumerate([MARKER_DIR, CLEAN_DIR, NOLEADER_DIR]):
            png = d / f"{stem}.png"
            if not png.exists():
                axes[row, col].text(0.5, 0.5, f"{stem} missing\n{d.name}", ha="center")
                continue
            bgr = cv2.imread(str(png))
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(rgb)
            axes[row, col].set_title(f"{stem}  {d.name.split('_')[-1]}", fontsize=9)
    plt.tight_layout()
    plt.show()

# show_triplets(start=1, n=4)